In [1]:
import torch
import torchvision
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.utils.data as data
import wandb
import os

In [2]:
from lightly.loss import NTXentLoss
from lightly.models.modules import SimCLRProjectionHead
from lightly.transforms.simclr_transform import SimCLRTransform

In [3]:
if 'google.colab' in str(get_ipython()):
    from google.colab import drive
    drive.mount('/content/drive')
    base_path = '/content/drive/MyDrive/ProjectSimCLR'
else:
    # Lokal auf deinem MacBook
    # Nutze absolute Pfade oder relative Pfade zu deinem Repository
    base_path = './' 

def save_checkpoint(model, optimizer, epoch, loss, run_id):
    torch.save({"epoch": epoch,
    "optimizer":optimizer,
    "model": model,
    "loss": loss,
    "wandb_run_id": run_id
    }, os.join(base_path, f"/models/{run_id}_epoch_{epoch}"))

def load_checkpoint(model, optimizer, path):
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    return checkpoint['epoch'], checkpoint['loss'], checkpoint['wandb_run_id']

In [4]:
class SimCLR(nn.Module): 
    def __init__(self, encoder, projection_head):
        super().__init__()
        self.encoder = encoder
        self.projection_head = projection_head

    def forward(self, x):
        x = self.encoder(x)
        x = torch.flatten(x, 1)
        x = self.projection_head(x)
        return x


In [5]:
resnet = torchvision.models.resnet18()
backbone = nn.Sequential(*list(resnet.children())[:-1])
projection_head = SimCLRProjectionHead(
    input_dim = 512,
    hidden_dim=512,
    output_dim=128
)
model = SimCLR(backbone, projection_head)

In [6]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
model.to(device)

SimCLR(
  (encoder): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)


In [12]:
hp = {
    "batch_size": 128,
    "pipeline_test" : True,
    "epochs":20, 
    "optim" : {
        "lr" : 0.06,
        "weight_decay": 1e-5
    }
    }

In [8]:
transform = SimCLRTransform(input_size=32, gaussian_blur=0.0)
dataset = datasets.CIFAR10(root="./data", train=True, transform=transform,  download=True)
if hp["pipeline_test"]:
    dataset = data.Subset(dataset, range(256))

/Users/hannesneumann/Repositories/SelfSupervisedLearning/.venv/lib/python3.13/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [9]:
dataloader = DataLoader(dataset, batch_size=hp["batch_size"], shuffle=True, drop_last=True,num_workers=4)


In [10]:
criterion = NTXentLoss()
optimizer = torch.optim.SGD(params=model.named_parameters(), **hp["optim"])

In [13]:
print("Starting Training ...")
run = wandb.init(project="simclr-cifar10", config=hp)
model = model.to(device)
for epoch in range(hp["epochs"]):
    total_loss = 0
    for batch in dataloader:
        x0, x1 = batch[0]
        x0 = x0.to(device)
        x1 = x1.to(device)
        z0 = model(x0)
        z1 = model(x1)

        loss = criterion(z0, z1)
        total_loss += loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    avg_loss = total_loss / len(dataloader)
    wandb.log({
    "loss": avg_loss,
    "epoch": epoch,
    "lr": optimizer.param_groups[0]['lr']
    })
    if epoch % 10 == 0 and epoch >= 20: 
        save_checkpoint(model, optimizer, loss, run.id)
    print(f"Epoch: {epoch}, Avg_loss: {avg_loss:.5f}")

Starting Training ...


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▃▅▆█
loss,█▆█▄▁
lr,▁▁▁▁▁
epoch,4
loss,5.4419
lr,0.06


Epoch: 0, Avg_loss: 5.45891
Epoch: 1, Avg_loss: 5.39040
Epoch: 2, Avg_loss: 5.40151
Epoch: 3, Avg_loss: 5.42764
Epoch: 4, Avg_loss: 5.40463
Epoch: 5, Avg_loss: 5.36728
Epoch: 6, Avg_loss: 5.38388
Epoch: 7, Avg_loss: 5.41882
Epoch: 8, Avg_loss: 5.37581
Epoch: 9, Avg_loss: 5.33949
Epoch: 10, Avg_loss: 5.34050
Epoch: 11, Avg_loss: 5.39854
Epoch: 12, Avg_loss: 5.29795
Epoch: 13, Avg_loss: 5.24301
Epoch: 14, Avg_loss: 5.33885
Epoch: 15, Avg_loss: 5.26160
Epoch: 16, Avg_loss: 5.29086
Epoch: 17, Avg_loss: 5.24837
Epoch: 18, Avg_loss: 5.27680
Epoch: 19, Avg_loss: 5.25982
